# 02 · Embeddings de las micrografías + clasificador (respaldo de la entrega)

**Kernel:** `Python-ML-DL` (trae PyTorch 2.3, torchvision, timm, scikit-learn, matplotlib).
**Recursos sugeridos:** cola `kura` (CPU) con **16 núcleos y 32 GB** (o una cola `nukwa-*` con GPU: el cuaderno usa la GPU si la ve). El número de núcleos que ve Python en el nodo (40) no es lo que *te asignaron*: la celda 1 imprime los núcleos asignados de verdad.

**Qué hace, en orden (cada celda termina con una comprobación; si falla, no sigas):**
1. Revisa recursos, librerías y carpetas.
2. Descomprime `datos/imagenes_v0.1.zip` (una sola vez) y comprueba que la tabla encuentra las imágenes.
3. Calcula el **vector de cada imagen** con dos modelos preentrenados congelados: `resnet50` (CNN) y `DINOv2 ViT-S/14` (transformer). Los pesos se bajan de Hugging Face (el nodo tiene acceso; `download.pytorch.org` está bloqueado, por eso no se usan los pesos de torchvision).
4. Entrena un clasificador ligero (regresión logística) con validación **por artículo** (`fold`) y escribe métricas, predicciones y matriz de confusión.

Salidas en `resultados/` (pequeñas; es lo que se descarga). Reanudable: si se corta, vuelve a ejecutar y retoma.

In [ ]:
# Celda 1: recursos y librerías
import os, sys, json, time, socket, platform
from pathlib import Path

BASE = Path(os.environ.get("N2P_BASE", Path.home() / "ondemand" / "Kabre_Nano2Predict"))
DATOS, SALIDA = BASE / "datos", BASE / "resultados"
SALIDA.mkdir(exist_ok=True)

asignados = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else os.cpu_count()
print("nodo:", socket.gethostname(), "| python:", platform.python_version())
print("núcleos que ve el nodo:", os.cpu_count(), "| núcleos ASIGNADOS a tu sesión:", asignados)

import numpy as np, pandas as pd
import torch, timm, sklearn
from PIL import Image
print("torch", torch.__version__, "| timm", timm.__version__, "| sklearn", sklearn.__version__)
dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
print("dispositivo:", dispositivo, torch.cuda.get_device_name(0) if dispositivo == "cuda" else "")
torch.set_num_threads(max(1, asignados))

import shutil
libre = shutil.disk_usage(BASE).free / 1e9
print(f"disco libre: {libre:.1f} GB")
assert (DATOS / "tabla_modelo.csv").exists(), "falta datos/tabla_modelo.csv"
assert (DATOS / "imagenes_v0.1.zip").exists() or (BASE / "imagenes").exists(), "falta datos/imagenes_v0.1.zip"
assert libre > 2, "poco disco libre"
print("OK celda 1")


In [ ]:
# Celda 2: descomprimir imágenes y cruzar con la tabla del modelo
import zipfile
from pathlib import PurePosixPath
IMG = BASE / "imagenes"
if not IMG.exists() or sum(1 for _ in IMG.rglob("*.jpg")) < 2000:
    zip_path = DATOS / "imagenes_v0.1.zip"
    assert zip_path.stat().st_size == 272306990, f"tamaño inesperado del zip: {zip_path.stat().st_size} (la subida pudo cortarse)"
    with zipfile.ZipFile(zip_path) as z:
        assert z.testzip() is None, "zip dañado"
        for zi in z.infolist():                    # el zip hecho en Windows guarda 'PMC123\archivo.jpg': se normaliza la barra
            nombre = zi.orig_filename.replace("\\", "/")
            p = PurePosixPath(nombre)
            if p.suffix.lower() == ".jpg" and len(p.parts) == 2 and ".." not in p.parts and not nombre.startswith("/"):
                destino = IMG / p.parts[0] / p.parts[1]      # solo imágenes pmcid/archivo; nada de rutas libres
                destino.parent.mkdir(parents=True, exist_ok=True)
                if not destino.exists():
                    destino.write_bytes(z.read(zi))
n_img = sum(1 for _ in IMG.rglob("*.jpg"))
print("imágenes en disco:", n_img)

t = pd.read_csv(DATOS / "tabla_modelo.csv", encoding="utf-8-sig")
t["ruta"] = t.apply(lambda r: IMG / str(r["pmcid"]) / str(r["image_file"]), axis=1)
t["existe"] = t["ruta"].map(lambda p: p.exists())
print("filas de la tabla:", len(t), "| con imagen:", int(t.existe.sum()), "| sin imagen:", int((~t.existe).sum()))
d = t[t.existe & t.clase_modelo.notna()].reset_index(drop=True)
print("usables (imagen + clase):", len(d), "| artículos:", d.pmcid.nunique())
print(d.clase_modelo.value_counts().to_string())
assert (d.groupby("pmcid").fold.nunique() == 1).all(), "un artículo está en dos folds"
assert len(d) > 1900, "faltan imágenes"
print("OK celda 2")


In [ ]:
# Celda 3: embeddings con modelos preentrenados congelados (reanudable)
from torch.utils.data import Dataset, DataLoader

MODELOS = {
    "resnet50": "resnet50.a1_in1k",                           # CNN
    "dinov2_vits14": "vit_small_patch14_dinov2.lvd142m",      # transformer (DINOv2)
}
os.environ.setdefault("HF_HOME", str(BASE / "hf_cache"))     # pesos en tu carpeta (unos 100-200 MB en total)
TAM = 224
MEDIA, DESV = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

def preparar(ruta):
    im = Image.open(ruta).convert("RGB")
    w, h = im.size
    lado = max(w, h)
    lienzo = Image.new("RGB", (lado, lado), (255, 255, 255))   # figura completa, rellenada a cuadrado (no se recortan paneles)
    lienzo.paste(im, ((lado - w) // 2, (lado - h) // 2))
    a = np.asarray(lienzo.resize((TAM, TAM), Image.BICUBIC), dtype=np.float32) / 255.0
    a = (a - np.array(MEDIA, dtype=np.float32)) / np.array(DESV, dtype=np.float32)
    return torch.from_numpy(a.transpose(2, 0, 1))

class DS(Dataset):
    def __init__(self, rutas): self.rutas = rutas
    def __len__(self): return len(self.rutas)
    def __getitem__(self, i): return preparar(self.rutas[i])

rutas = [str(p) for p in d.ruta]
emb = {}
for corto, nombre in MODELOS.items():
    destino = SALIDA / f"emb_{corto}.npy"
    if destino.exists() and np.load(destino).shape[0] == len(d):
        emb[corto] = np.load(destino); print(corto, "ya calculado", emb[corto].shape); continue
    kw = {"img_size": TAM} if "vit" in nombre else {}
    modelo = timm.create_model(nombre, pretrained=True, num_classes=0, **kw).eval().to(dispositivo)
    dl = DataLoader(DS(rutas), batch_size=32, num_workers=min(8, asignados), shuffle=False)
    out, t0 = [], time.time()
    with torch.inference_mode():
        for i, xb in enumerate(dl):
            out.append(modelo(xb.to(dispositivo)).float().cpu().numpy())
            if i % 10 == 0: print(f"{corto}: {min((i+1)*32, len(rutas))}/{len(rutas)} ({time.time()-t0:.0f}s)", flush=True)
    emb[corto] = np.concatenate(out)
    assert emb[corto].shape[0] == len(d) and np.isfinite(emb[corto]).all()
    np.save(destino, emb[corto])
    print(corto, "listo", emb[corto].shape, f"{time.time()-t0:.0f}s")
d[["pmcid", "image_file", "clase_modelo", "fold"]].to_csv(SALIDA / "indice_embeddings.csv", index=False, encoding="utf-8")
print("OK celda 3")


In [ ]:
# Celda 4: clasificador ligero con validación por artículo (fold)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

y = d.clase_modelo.values
folds = d.fold.astype(int).values
clases = sorted(set(y))
base = pd.Series(y).value_counts(normalize=True).max()
resultados = {"n_imagenes": int(len(d)), "n_articulos": int(d.pmcid.nunique()), "clases": clases, "evaluacion": "5 particiones por articulo (columna fold)", "linea_base_clase_mayoritaria": round(float(base), 4), "modelos": {}}

for corto, X in emb.items():
    pred = np.empty(len(d), dtype=object)
    prob = np.zeros(len(d))
    for k in sorted(set(folds)):
        tr, te = folds != k, folds == k
        clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, C=0.1, class_weight="balanced", random_state=0))
        clf.fit(X[tr], y[tr])
        pred[te] = clf.predict(X[te])
        prob[te] = clf.predict_proba(X[te]).max(axis=1)
    acc, f1 = accuracy_score(y, pred), f1_score(y, pred, average="macro")
    cm = confusion_matrix(y, pred, labels=clases)
    resultados["modelos"][corto] = {"accuracy": round(float(acc), 4), "f1_macro": round(float(f1), 4), "matriz_confusion": cm.tolist()}
    pd.DataFrame({"pmcid": d.pmcid, "image_file": d.image_file, "real": y, "prediccion": pred, "confianza": prob.round(4), "fold": folds}).to_csv(SALIDA / f"predicciones_{corto}.csv", index=False, encoding="utf-8")
    print(f"{corto}: exactitud {acc:.3f} | F1 macro {f1:.3f} | (línea base {base:.3f})")
    print(pd.DataFrame(cm, index=clases, columns=clases).to_string(), "\n")

with open(SALIDA / "metricas_embeddings.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)
print("Archivos en", SALIDA)
for p in sorted(SALIDA.iterdir()): print(" ", p.name, f"{p.stat().st_size/1e6:.1f} MB")
print("OK celda 4. Descarga metricas_embeddings.json y predicciones_*.csv y pégaselos a Claude.")
